<h3>Modelling

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
from evaluate_binary_classifier import *

In [3]:
PROJ_ROOT = Path().resolve().parents[0]
DATA_DIR = PROJ_ROOT / "data"
INTERIM_DATA_DIR = DATA_DIR / "interim"

In [4]:
X_test = pd.read_pickle(INTERIM_DATA_DIR / "X_test_processed.pkl")
X_train = pd.read_pickle(INTERIM_DATA_DIR / "X_train_sampled.pkl")
y_test = pd.read_pickle(INTERIM_DATA_DIR / "y_test_processed.pkl")
y_train = pd.read_pickle(INTERIM_DATA_DIR / "y_train_sampled.pkl")

Here we will try several different binary classification models.

<h4>Logistic Regression

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV

In [6]:
# Base model:

lr_model = LogisticRegression()

lr_model.fit(X_train,y_train)

y_pred = lr_model.predict(X_test)
y_proba = lr_model.predict_proba(X_test)[:,1]

lr_model_base = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)


===== Binary Classification Evaluation =====

Accuracy: 0.8913
Balanced Accuracy: 0.7866
Precision: 0.3656
Recall: 0.6639
F1 Score: 0.4715
Matthews Corrcoef: 0.4397
ROC AUC: 0.8874
Log Loss: 0.3239

Confusion Matrix:
[[9715  970]
 [ 283  559]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94     10685
           1       0.37      0.66      0.47       842

    accuracy                           0.89     11527
   macro avg       0.67      0.79      0.71     11527
weighted avg       0.93      0.89      0.91     11527



In [ ]:
# Hyperparameter tuning and K-fold validation

lr_model = LogisticRegression(max_iter=1000, random_state=13)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=13
)

param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs"],
    "class_weight": [None, "balanced"]
}

grid_search_lr = GridSearchCV(
    estimator=lr_model,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_search_lr.fit(X_train, y_train)

# Training score
print("Best 5-fold CV F1-score:", grid_search_lr.best_score_)
print("Best parameters:")
print(grid_search_lr.best_params_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best 5-fold CV F1-score: 0.8887995024006449
Best parameters:
{'C': 1, 'class_weight': None, 'penalty': 'l2', 'solver': 'lbfgs'}


c:\Users\cochr\Data Science Projects\lA82lBNDpo9AlAnK\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [8]:
# Test score:
best_lr = grid_search.best_estimator_
y_pred = best_lr.predict(X_test)
y_proba = best_lr.predict_proba(X_test)[:,1]

best_lr_evaluate = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)


===== Binary Classification Evaluation =====

Accuracy: 0.8913
Balanced Accuracy: 0.7866
Precision: 0.3656
Recall: 0.6639
F1 Score: 0.4715
Matthews Corrcoef: 0.4397
ROC AUC: 0.8874
Log Loss: 0.3239

Confusion Matrix:
[[9715  970]
 [ 283  559]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94     10685
           1       0.37      0.66      0.47       842

    accuracy                           0.89     11527
   macro avg       0.67      0.79      0.71     11527
weighted avg       0.93      0.89      0.91     11527



In [9]:
# The test F1-score is 47% (not great), but the 5 fold CV F1-score is 89% (the project aim is: "Hit 81% or above F1-score by evaluating with 5-fold cross validation and reporting the average performance score.")

# Let's try some other model types to get the test F1 score up.

<h4>Decision Trees

In [10]:
from sklearn.tree import DecisionTreeClassifier

In [11]:
dt_model = DecisionTreeClassifier()

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
y_proba = dt_model.predict_proba(X_test)[:,1]

dt_model_base = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)


===== Binary Classification Evaluation =====

Accuracy: 0.9008
Balanced Accuracy: 0.7271
Precision: 0.3725
Recall: 0.5238
F1 Score: 0.4353
Matthews Corrcoef: 0.3893
ROC AUC: 0.7276
Log Loss: 3.5742

Confusion Matrix:
[[9942  743]
 [ 401  441]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.93      0.95     10685
           1       0.37      0.52      0.44       842

    accuracy                           0.90     11527
   macro avg       0.67      0.73      0.69     11527
weighted avg       0.92      0.90      0.91     11527



In [13]:
# Hyperparameter tuning and K-fold validation

dt_model = DecisionTreeClassifier()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=13
)

param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10, 15, None],
    "min_samples_split": [2, 5, 10, 20, 50],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "max_features": [None, "sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

grid_search_dt = GridSearchCV(
    estimator=dt_model,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_search_dt.fit(X_train, y_train)

# Training score
print("Best 5-fold CV F1-score:", grid_search_dt.best_score_)
print("Best parameters:")
print(grid_search_dt.best_params_)

Fitting 5 folds for each of 1800 candidates, totalling 9000 fits
Best 5-fold CV F1-score: 0.9336125516405375
Best parameters:
{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 15, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [14]:
# Test score:
best_dt = grid_search_dt.best_estimator_
y_pred = best_dt.predict(X_test)
y_proba = best_dt.predict_proba(X_test)[:,1]

best_dt_evaluate = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)


===== Binary Classification Evaluation =====

Accuracy: 0.8950
Balanced Accuracy: 0.7836
Precision: 0.3747
Recall: 0.6532
F1 Score: 0.4762
Matthews Corrcoef: 0.4428
ROC AUC: 0.8002
Log Loss: 1.8452

Confusion Matrix:
[[9767  918]
 [ 292  550]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94     10685
           1       0.37      0.65      0.48       842

    accuracy                           0.90     11527
   macro avg       0.67      0.78      0.71     11527
weighted avg       0.93      0.90      0.91     11527



In [ ]:
# HPT got the test F1-score from 44% to 48%. The training F1-score is high: 93%

<h4>Random Forests

In [5]:
from sklearn.ensemble import RandomForestClassifier

In [6]:
rfc_model = RandomForestClassifier()

rfc_model.fit(X_train, y_train)

y_pred = rfc_model.predict(X_test)
y_proba = rfc_model.predict_proba(X_test)[:,1]

rfc_model_base = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)


===== Binary Classification Evaluation =====

Accuracy: 0.9192
Balanced Accuracy: 0.7508
Precision: 0.4564
Recall: 0.5534
F1 Score: 0.5003
Matthews Corrcoef: 0.4593
ROC AUC: 0.9230
Log Loss: 0.2374

Confusion Matrix:
[[10130   555]
 [  376   466]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.95      0.96     10685
           1       0.46      0.55      0.50       842

    accuracy                           0.92     11527
   macro avg       0.71      0.75      0.73     11527
weighted avg       0.93      0.92      0.92     11527



In [16]:
# Hyperparameter tuning and K-fold validation

rfc_model = RandomForestClassifier()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=13
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [20, 30],
    "min_samples_split": [2, 5, 7],
    "min_samples_leaf": [2, 5, 7],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"],
    "bootstrap": [True] # Whether each tree trains on bootstrap samples (each tree is trained on a random sample of the training rows drawn with replacement (the same row can be drawn multiple times))
}

grid_search_rfc = GridSearchCV(
    estimator=rfc_model,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_search_rfc.fit(X_train, y_train)

# Training score
print("Best 5-fold CV F1-score:", grid_search_rfc.best_score_)
print("Best parameters:")
print(grid_search_rfc.best_params_)

Fitting 5 folds for each of 144 candidates, totalling 720 fits
Best 5-fold CV F1-score: 0.9522389794358193
Best parameters:
{'bootstrap': True, 'class_weight': None, 'max_depth': 30, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}


In [17]:
# Test score:
best_rfc = grid_search_rfc.best_estimator_
y_pred = best_rfc.predict(X_test)
y_proba = best_rfc.predict_proba(X_test)[:,1]

best_rfc_evaluate = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)


===== Binary Classification Evaluation =====

Accuracy: 0.9147
Balanced Accuracy: 0.8019
Precision: 0.4444
Recall: 0.6698
F1 Score: 0.5343
Matthews Corrcoef: 0.5020
ROC AUC: 0.9327
Log Loss: 0.1841

Confusion Matrix:
[[9980  705]
 [ 278  564]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.93      0.95     10685
           1       0.44      0.67      0.53       842

    accuracy                           0.91     11527
   macro avg       0.71      0.80      0.74     11527
weighted avg       0.93      0.91      0.92     11527



In [19]:
# 3% increase in test F1-score to 53%

# High CV F1-score of 95%

<h4>K-Nearest Neighbours

In [20]:
from sklearn.neighbors import KNeighborsClassifier

In [21]:
knn_model = KNeighborsClassifier()

knn_model.fit(X_train, y_train)

y_pred = knn_model.predict(X_test)
y_proba = knn_model.predict_proba(X_test)[:,1]

knn_model_base = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)


===== Binary Classification Evaluation =====

Accuracy: 0.8839
Balanced Accuracy: 0.8066
Precision: 0.3543
Recall: 0.7162
F1 Score: 0.4741
Matthews Corrcoef: 0.4499
ROC AUC: 0.8628
Log Loss: 1.7185

Confusion Matrix:
[[9586 1099]
 [ 239  603]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.90      0.93     10685
           1       0.35      0.72      0.47       842

    accuracy                           0.88     11527
   macro avg       0.66      0.81      0.70     11527
weighted avg       0.93      0.88      0.90     11527



In [26]:
# Hyperparameter tuning and K-fold validation

knn_model = KNeighborsClassifier()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=13
)

param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
    "weights": ["uniform", "distance"],
    "metric": ["minkowski", "manhattan", "euclidean"],
    "p": [1, 2] #Controls the distance formula when the metric is uses minkowski distance...
}

grid_search_knn = GridSearchCV(
    estimator=knn_model,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_search_knn.fit(X_train, y_train)

# Training score
print("Best 5-fold CV F1-score:", grid_search_knn.best_score_)
print("Best parameters:")
print(grid_search_knn.best_params_)

# Test score:
best_knn = grid_search_knn.best_estimator_
y_pred = best_knn.predict(X_test)
y_proba = best_knn.predict_proba(X_test)[:,1]

best_knn_evaluate = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)

Fitting 5 folds for each of 96 candidates, totalling 480 fits
Best 5-fold CV F1-score: 0.9445665782854922
Best parameters:
{'metric': 'minkowski', 'n_neighbors': 3, 'p': 1, 'weights': 'distance'}

===== Binary Classification Evaluation =====

Accuracy: 0.8998
Balanced Accuracy: 0.7627
Precision: 0.3821
Recall: 0.6021
F1 Score: 0.4675
Matthews Corrcoef: 0.4283
ROC AUC: 0.8403
Log Loss: 2.0830

Confusion Matrix:
[[9865  820]
 [ 335  507]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.92      0.94     10685
           1       0.38      0.60      0.47       842

    accuracy                           0.90     11527
   macro avg       0.67      0.76      0.71     11527
weighted avg       0.92      0.90      0.91     11527



In [28]:
# No real improvement from hypertuning. CV F1-score is high (94%), but the test F1-score is below 50%. Not great.

<h4> Support Vector Machine (SVM)

In [29]:
from sklearn.svm import SVC

In [32]:
svc_model = SVC(probability=True)

svc_model.fit(X_train, y_train)

y_pred = svc_model.predict(X_test)
y_proba = svc_model.predict_proba(X_test)[:,1]

svc_model_base = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)


===== Binary Classification Evaluation =====

Accuracy: 0.8974
Balanced Accuracy: 0.8123
Precision: 0.3894
Recall: 0.7126
F1 Score: 0.5036
Matthews Corrcoef: 0.4775
ROC AUC: 0.9161
Log Loss: 0.2393

Confusion Matrix:
[[9744  941]
 [ 242  600]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.91      0.94     10685
           1       0.39      0.71      0.50       842

    accuracy                           0.90     11527
   macro avg       0.68      0.81      0.72     11527
weighted avg       0.93      0.90      0.91     11527



In [35]:
# 2 minute run time. Hyperparameter tuning with GridSearchCV will be hard...

<h4>XGBoost

In [1]:
from xgboost import XGBClassifier

In [7]:
xgb_model = XGBClassifier()

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:,1]

xgb_model_base = evaluate_binary_classifier(
    y_test=y_test,
    y_pred=y_pred,
    y_proba=y_proba
)

# Return/store the classifier's base parameters
xgb_base_params = xgb_model.get_params()
xgb_base_params


===== Binary Classification Evaluation =====

Accuracy: 0.9293
Balanced Accuracy: 0.7480
Precision: 0.5154
Recall: 0.5356
F1 Score: 0.5253
Matthews Corrcoef: 0.4873
ROC AUC: 0.9324
Log Loss: 0.1623

Confusion Matrix:
[[10261   424]
 [  391   451]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96     10685
           1       0.52      0.54      0.53       842

    accuracy                           0.93     11527
   macro avg       0.74      0.75      0.74     11527
weighted avg       0.93      0.93      0.93     11527



{'objective': 'binary:logistic',
 'base_score': None,
 'booster': None,
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': None,
 'device': None,
 'early_stopping_rounds': None,
 'enable_categorical': False,
 'eval_metric': None,
 'feature_types': None,
 'feature_weights': None,
 'gamma': None,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': None,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': None,
 'max_leaves': None,
 'min_child_weight': None,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': None,
 'n_jobs': None,
 'num_parallel_tree': None,
 'random_state': None,
 'reg_alpha': None,
 'reg_lambda': None,
 'sampling_method': None,
 'scale_pos_weight': None,
 'subsample': None,
 'tree_method': None,
 'validate_parameters': None,
 'verbosity': None}

In [ ]:
# Hyperparameter tuning and K-fold validation

xgb_model = XGBClassifier()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=13
)

param_grid = {
    "n_estimators":[100, 500, 1000, 4000, 10000],
    "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    "subsample" : [0.7],
    "max_depth" : [7]
    
}

grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_search_xgb.fit(X_train, y_train)

# Training score
print("Best 5-fold CV F1-score:", grid_search_xgb.best_score_)
print("Best parameters:")
print(grid_search_xgb.best_params_)

# Test score:
best_xgb = grid_search_xgb.best_estimator_
y_pred = best_xgb.predict(X_test)
y_proba = best_xgb.predict_proba(X_test)[:,1]

best_xgb_evaluate = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best 5-fold CV F1-score: 0.9626533407310406
Best parameters:
{'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 4000, 'subsample': 0.7}

===== Binary Classification Evaluation =====

Accuracy: 0.9294
Balanced Accuracy: 0.7327
Precision: 0.5171
Recall: 0.5024
F1 Score: 0.5096
Matthews Corrcoef: 0.4717
ROC AUC: 0.9335
Log Loss: 0.1653

Confusion Matrix:
[[10290   395]
 [  419   423]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96     10685
           1       0.52      0.50      0.51       842

    accuracy                           0.93     11527
   macro avg       0.74      0.73      0.74     11527
weighted avg       0.93      0.93      0.93     11527



In [9]:
# The hypertuned version has a worse test F1-score than the base?

The consistently higher CV F1 score compared to the test F1-score among the different models implies overfitting?

In [ ]:
# Is there a way to fix this? Re-do with larger param grid but use randomised search...

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

In [19]:
# Hyperparameter tuning and K-fold validation

xgb_model = XGBClassifier()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=13
)

param_distributions = {
    "n_estimators": randint(100, 700),
    "learning_rate": loguniform(0.01, 0.12),

    "max_depth": randint(2, 6),
    "min_child_weight": randint(3, 21),
    "gamma": uniform(0, 1),

    "subsample": uniform(0.6, 0.35),
    "colsample_bytree": uniform(0.6, 0.35),

    "reg_alpha": loguniform(1e-4, 2),
    "reg_lambda": loguniform(1, 30)
}

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=225,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=13,
    return_train_score=True,
    refit=True
)

random_search.fit(X_train, y_train)

# Training score
print("Best 5-fold CV F1-score:", random_search.best_score_)
print("Best parameters:")
print(random_search.best_params_)

# Test score:
best_xgb = random_search.best_estimator_
y_pred = best_xgb.predict(X_test)
y_proba = best_xgb.predict_proba(X_test)[:,1]

best_xgb_evaluate = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)

Fitting 5 folds for each of 225 candidates, totalling 1125 fits
Best 5-fold CV F1-score: 0.9597015996151479
Best parameters:
{'colsample_bytree': np.float64(0.7068258974658037), 'gamma': np.float64(0.30060668803582336), 'learning_rate': np.float64(0.06753700296765182), 'max_depth': 5, 'min_child_weight': 4, 'n_estimators': 680, 'reg_alpha': np.float64(0.00039223485954332447), 'reg_lambda': np.float64(1.022814750947815), 'subsample': np.float64(0.6902399852768674)}

===== Binary Classification Evaluation =====

Accuracy: 0.9279
Balanced Accuracy: 0.7407
Precision: 0.5063
Recall: 0.5214
F1 Score: 0.5138
Matthews Corrcoef: 0.4749
ROC AUC: 0.9361
Log Loss: 0.1561

Confusion Matrix:
[[10257   428]
 [  403   439]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96     10685
           1       0.51      0.52      0.51       842

    accuracy                           0.93     11527
   macro avg       0.73      0.74      0

I may have done the preprocessing and sampling incorrectly. We are using Kfold validation here, so the preprocessing and sampling needs to be done on each fold separately within a pipeline to reduce data leakage and therefore mitigate overfitting.